In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
DATA = Path("data")

## The architecture

In [ ]:
def lenet5():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 1)),
        keras.layers.Conv2D(6, kernel_size=(5, 5), strides=1, padding="valid", activation="tanh"),
        keras.layers.AveragePooling2D(pool_size=(2, 2), strides=2),
        keras.layers.Conv2D(16, kernel_size=(5, 5), strides=1, padding="valid", activation="tanh"),
        keras.layers.AveragePooling2D(pool_size=(2, 2), strides=2),
        keras.layers.Flatten(),
        keras.layers.Dense(120, activation="tanh"),
        keras.layers.Dense(84, activation="tanh"),
        keras.layers.Dense(10, activation="softmax")])

model = lenet5()
model.summary()
rows = [dict(layer=l.name, shape=str(l.output.shape[1:]), params=l.count_params()) for l in model.layers]
table = pd.DataFrame(rows)
print(table.to_string(index=False))
print("total:", model.count_params())
table.to_csv(DATA / "lenet_summary.csv", index=False)

## Parameters by hand: (filter height x width x input channels + 1 bias) x filters

In [ ]:
print("conv 1:", (5 * 5 * 1 + 1) * 6)
print("conv 2:", (5 * 5 * 6 + 1) * 16)
print("dense 120:", 400 * 120 + 120, " dense 84:", 120 * 84 + 84, " output:", 84 * 10 + 10)
print("the paper's layers C1, S2, C3, S4, C5, F6:", 156 + 12 + 1516 + 32 + 48120 + 10164)

## Training on MNIST
MNIST digits are 28 x 28; LeNet-5 expects 32 x 32, so we add 2 pixels of zeros on each side, as the paper's
input was larger than the digits. Pixels are scaled to 0-1. 10 epochs, 3 seeds.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = np.pad(x_train, ((0, 0), (2, 2), (2, 2)))[..., None] / 255.0
x_test = np.pad(x_test, ((0, 0), (2, 2), (2, 2)))[..., None] / 255.0
print(x_train.shape, x_test.shape)

In [ ]:
results, histories = [], []
for seed in (1, 2, 3):
    keras.utils.set_random_seed(seed)
    m = lenet5()
    m.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    h = m.fit(x_train, y_train, epochs=10, batch_size=128, validation_split=0.1, verbose=0)
    acc = m.evaluate(x_test, y_test, verbose=0)[1]
    results.append(acc)
    histories.append(pd.DataFrame(h.history).assign(seed=seed, epoch=np.arange(1, 11)))
    print(f"seed {seed}: test accuracy {acc:.4f}")
print(f"mean test accuracy {np.mean(results):.4f}, test error {100 * (1 - np.mean(results)):.2f}%")
pd.concat(histories).to_csv(DATA / "lenet_history.csv", index=False)